# Блок 4. Построение ML моделей

## Описание блока:
В данном блоке будут обучены ML модели с применением кросс валидации и подбора лучших гиперпараметров. Подробное текстовое описания получившегося результата будет в  [Открыть тетрадку "Part 0"](Part%200.%20Research%20results.ipynb)

In [1]:
# импорт основных библиотек блока
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from all_functions.datasets_load import load_all_datasets
from all_functions.datasets_transfom import *
from all_functions.func_for_ml import * 


/home/danila/mainpk/masters_work/Project_Financial_assets_and_ML/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# фиксация сида на будущее
RANDOM_STATE = 654321

In [3]:
# загрузка датасетов
crypt_Ethereum, crypt_BTC, futures_Brent_LCOK5, \
futures_WTI_CLJ5, share_metal_jiangxi_copper, \
share_metal_baoshan_iron, share_oil_petrochina_hk, \
share_oil_equinor_oslo, spot_preciouse_metal_AG, spot_preciouse_metal_AU = load_all_datasets()

In [4]:
# трансформация датасетов
crypt_Ethereum = transform_data(crypt_Ethereum)
crypt_BTC = transform_data(crypt_BTC)

futures_WTI_CLJ5 = transform_data(futures_WTI_CLJ5)
futures_Brent_LCOK5 = transform_data(futures_Brent_LCOK5)

share_metal_jiangxi_copper  = transform_data(share_metal_jiangxi_copper)
share_metal_baoshan_iron  = transform_data(share_metal_baoshan_iron)

share_oil_petrochina_hk  = transform_data(share_oil_petrochina_hk)
share_oil_equinor_oslo  = transform_data(share_oil_equinor_oslo)

spot_preciouse_metal_AU  = transform_data(spot_preciouse_metal_AU)
spot_preciouse_metal_AG = transform_data(spot_preciouse_metal_AG)

## Построение ML моделей

In [5]:
all_data = {'crypt_Ethereum': crypt_Ethereum, 
 'crypt_BTC':crypt_BTC,
 'futures_Brent_LCOK5':futures_Brent_LCOK5,
'futures_WTI_CLJ5': futures_WTI_CLJ5,
'share_metal_jiangxi_copper': share_metal_jiangxi_copper, 
'share_metal_baoshan_iron': share_metal_baoshan_iron,
 'share_oil_petrochina_hk': share_oil_petrochina_hk, 
'share_oil_equinor_oslo': share_oil_equinor_oslo,
 'spot_preciouse_metal_AG': spot_preciouse_metal_AG,
 'spot_preciouse_metal_AU': spot_preciouse_metal_AU}


In [6]:
# Создание датасета для записи результатов обучения
all_reault = pd.DataFrame()

name_ds = []
for i, _ in all_data.items():
    name_ds.append(i)
all_reault['dataset_name'] = name_ds

# фиксация в общем датасете средней цены, для наглядности диапазона цен

mean_ds = []
for i, n in all_data.items():
    mean_ds.append(n.mean()['Цена'])
all_reault['dataset_mean_price'] = mean_ds

all_reault

,dataset_name,dataset_mean_price
0,crypt_Ethereum,1264.224518
1,crypt_BTC,24007.290015
2,futures_Brent_LCOK5,68.036667
3,futures_WTI_CLJ5,63.524965
4,share_metal_jiangxi_copper,10.239784
5,share_metal_baoshan_iron,5.255690
6,share_oil_petrochina_hk,3.622050
7,share_oil_equinor_oslo,187.087574
8,spot_preciouse_metal_AG,20.814301
9,spot_preciouse_metal_AU,1677.948521


In [7]:
def objective_RFregresson(trial, data):
    
    # --- Гиперпараметры для Random Forest ---
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])

    # --- Разделение данных ---
    train_raw, valid_raw, _ = split_data(data)
    tail_len = 30

    train = upgrade_dataset(train_raw)
    valid_with_tail = pd.concat([train_raw.tail(tail_len), valid_raw])
    valid_with_tail = upgrade_dataset(valid_with_tail)
    valid = valid_with_tail.loc[valid_raw.index]

    X_train, y_train = return_x_y(train)
    X_valid, y_valid = return_x_y(valid)

    # --- Модель ---
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    return rmse

In [8]:
%%time

result_best_RF = []

for i, n in all_data.items():

    line_length = 150
    print('-' * line_length)
    print(f'Baseline датасета: {i}'.center(line_length))
    print('-' * line_length)

    optuna.logging.set_verbosity(optuna.logging.ERROR)
    study = optuna.create_study(direction="minimize")  # ничего лишнего
    study.optimize(partial(objective_RFregresson, data=n), n_trials=15)

    print("Лучшие параметры:", study.best_params)
    print("Лучшее RMSE:", study.best_value)

    result_best_RF.append(study.best_value)
    

all_reault['RF_result'] = result_best_RF

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                          Baseline датасета: crypt_Ethereum                                                           
------------------------------------------------------------------------------------------------------------------------------------------------------
Лучшие параметры: {'n_estimators': 184, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt'}
Лучшее RMSE: 126.24668513910166
------------------------------------------------------------------------------------------------------------------------------------------------------
                                                             Baseline датасета: crypt_BTC                                                             
------------------------------------------------------------------------------------

In [9]:
def objective_gbr(trial, data):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5)
    }

    train_raw, valid_raw, _ = split_data(data)
    tail_len = 30

    train = upgrade_dataset(train_raw)
    valid_with_tail = pd.concat([train_raw.tail(tail_len), valid_raw])
    valid_with_tail = upgrade_dataset(valid_with_tail)
    valid = valid_with_tail.loc[valid_raw.index]

    X_train, y_train = return_x_y(train)
    X_valid, y_valid = return_x_y(valid)

    model = GradientBoostingRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    return rmse

In [10]:
%%time

result_best_gbr = []

for i, n in all_data.items():

    line_length = 150
    print('-' * line_length)
    print(f'Baseline датасета: {i}'.center(line_length))
    print('-' * line_length)

    optuna.logging.set_verbosity(optuna.logging.ERROR)
    study = optuna.create_study(direction="minimize")  # ничего лишнего
    study.optimize(partial(objective_gbr, data=n), n_trials=15)

    print("Лучшие параметры:", study.best_params)
    print("Лучшее RMSE:", study.best_value)

    result_best_gbr.append(study.best_value)
    

all_reault['GBR_result'] = result_best_gbr

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                          Baseline датасета: crypt_Ethereum                                                           
------------------------------------------------------------------------------------------------------------------------------------------------------
Лучшие параметры: {'n_estimators': 239, 'learning_rate': 0.04063717639143676, 'max_depth': 4, 'subsample': 0.6303381907682958, 'min_samples_split': 7, 'min_samples_leaf': 5}
Лучшее RMSE: 120.7240964726063
------------------------------------------------------------------------------------------------------------------------------------------------------
                                                             Baseline датасета: crypt_BTC                                                             
----------------------------------------

In [11]:
def objective_lgbm(trial, data):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 7, 255),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
        # "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        # "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0)
    }

    train_raw, valid_raw, _ = split_data(data)
    tail_len = 30

    train = upgrade_dataset(train_raw)
    valid_with_tail = pd.concat([train_raw.tail(tail_len), valid_raw])
    valid_with_tail = upgrade_dataset(valid_with_tail)
    valid = valid_with_tail.loc[valid_raw.index]

    X_train, y_train = return_x_y(train)
    X_valid, y_valid = return_x_y(valid)

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    return rmse

In [12]:
%%time

result_best_lgbm = []

for i, n in all_data.items():

    line_length = 150
    print('-' * line_length)
    print(f'Baseline датасета: {i}'.center(line_length))
    print('-' * line_length)

    optuna.logging.set_verbosity(optuna.logging.ERROR)
    study = optuna.create_study(direction="minimize")  # ничего лишнего
    study.optimize(partial(objective_gbr, data=n), n_trials=15)

    print("Лучшие параметры:", study.best_params)
    print("Лучшее RMSE:", study.best_value)

    result_best_lgbm.append(study.best_value)
    

all_reault['LGBM_result'] = result_best_lgbm

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                          Baseline датасета: crypt_Ethereum                                                           
------------------------------------------------------------------------------------------------------------------------------------------------------
Лучшие параметры: {'n_estimators': 89, 'learning_rate': 0.14838244465395609, 'max_depth': 4, 'subsample': 0.593966150600739, 'min_samples_split': 6, 'min_samples_leaf': 1}
Лучшее RMSE: 131.56468480609198
------------------------------------------------------------------------------------------------------------------------------------------------------
                                                             Baseline датасета: crypt_BTC                                                             
-----------------------------------------

In [13]:
all_reault

,dataset_name,dataset_mean_price,RF_result,GBR_result,LGBM_result
0,crypt_Ethereum,1264.224518,126.246685,120.724096,131.564685
1,crypt_BTC,24007.290015,3394.572377,3064.070778,3070.375141
2,futures_Brent_LCOK5,68.036667,1.762227,1.674554,1.725623
3,futures_WTI_CLJ5,63.524965,1.625811,1.503803,1.489014
4,share_metal_jiangxi_copper,10.239784,0.448139,0.408249,0.416273
5,share_metal_baoshan_iron,5.255690,0.142422,0.131370,0.141907
6,share_oil_petrochina_hk,3.622050,1.268672,1.262738,1.248529
7,share_oil_equinor_oslo,187.087574,8.106792,7.271537,6.973421
8,spot_preciouse_metal_AG,20.814301,1.740925,1.524666,1.549896
9,spot_preciouse_metal_AU,1677.948521,301.950555,299.429745,295.770246
